In [1]:
# ---------------------------------------------------------------
# SETUP: run this cell first
# ---------------------------------------------------------------
# AI disclosure: this cell (and the cleaning cells that follow) was
# drafted with help from Claude (Anthropic), using only functions
# covered in Lectures 06-07. We ran it on our cluster and checked
# the outputs ourselves.
# ---------------------------------------------------------------

# Load the PySpark tools used in the cleaning steps below
from pyspark.sql import Row                          # to build the cleaning log table
from pyspark.sql import functions as F               # built-in functions (min, max, when)
from pyspark.sql.functions import col, expr, lit, to_date, regexp_extract, input_file_name

# Where the data lives and where cleaned results are saved
BUCKET = "gs://ba843_michaels_part"                  # change to "gs://qst5project" for the shared bucket
DATA = BUCKET + "/statsbomb/data/"                   # raw StatsBomb JSON files
OUT = BUCKET + "/statsbomb/clean/"                   # cleaned parquet files go here

# Test mode cleans only 2 tournaments (~128 matches) so each cell runs quickly.
# Set TEST_MODE = False for the final run on every competition.
TEST_MODE = True
TEST_COMPETITIONS = ["FIFA World Cup", "Women's World Cup"]   # one men's and one women's tournament
TEST_SEASONS = ["2022", "2023"]

# Save test results in a separate folder so they don't mix with the final run
if TEST_MODE:
    OUT = OUT + "test/"

# Empty list: each cleaning step adds a row (rows before/after),
# which becomes the before/after summary table at the end
cleaning_log = []

# Confirm the paths are correct
print("Reading from:", DATA)
print("Writing to:  ", OUT)

Reading from: gs://ba843_michaels_part/statsbomb/data/
Writing to:   gs://ba843_michaels_part/statsbomb/clean/test/


In [2]:
# ---------------------------------------------------------------
# STEP 1: Competitions
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using the read pattern from Notebook 01. We ran it on our cluster
# and checked the output ourselves.
# ---------------------------------------------------------------
# competitions.json lists every competition and season in the dataset.
# Same read pattern as Notebook 01 (format + load). The multiLine option
# is needed because each StatsBomb file is one big JSON array,
# not one record per line like the flight data in class.

comps = spark.read.format("json")\
  .option("multiLine", "true")\
  .load(DATA + "competitions.json")

# Show the competitions available, sorted by name and season
comps.select("competition_id", "season_id", "competition_name", "competition_gender", "season_name")\
  .orderBy("competition_name", "season_name")\
  .show(100, False)

+--------------+---------+-----------------------+------------------+-----------+
|competition_id|season_id|competition_name       |competition_gender|season_name|
+--------------+---------+-----------------------+------------------+-----------+
|9             |27       |1. Bundesliga          |male              |2015/2016  |
|9             |281      |1. Bundesliga          |male              |2023/2024  |
|1267          |107      |African Cup of Nations |male              |2023       |
|16            |276      |Champions League       |male              |1970/1971  |
|16            |71       |Champions League       |male              |1971/1972  |
|16            |277      |Champions League       |male              |1972/1973  |
|16            |76       |Champions League       |male              |1999/2000  |
|16            |44       |Champions League       |male              |2003/2004  |
|16            |37       |Champions League       |male              |2004/2005  |
|16            |

In [3]:
# ---------------------------------------------------------------
# STEP 2: Matches - load the raw data
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using the read pattern from Notebook 01. We ran it on our cluster
# and checked the output ourselves.
# ---------------------------------------------------------------
# Each competition-season has its own match file, stored as
# matches/<competition_id>/<season_id>.json
# The * wildcards read every folder and every file at once.

matches_raw = spark.read.format("json")\
  .option("multiLine", "true")\
  .load(DATA + "matches/*/*.json")

# Look at the structure: many columns are nested (structs inside structs),
# e.g. competition.competition_name, home_team.home_team_name
matches_raw.printSchema()

# How many matches and columns we start with
print(matches_raw.count(), "matches,", len(matches_raw.columns), "top-level columns")

root
 |-- away_score: long (nullable = true)
 |-- away_team: struct (nullable = true)
 |    |-- away_team_gender: string (nullable = true)
 |    |-- away_team_group: string (nullable = true)
 |    |-- away_team_id: long (nullable = true)
 |    |-- away_team_name: string (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- managers: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- country: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- dob: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- nickname: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- competition_id: long (nullable = true)
 |    |-- competition_nam

3961 matches, 18 top-level columns


In [4]:
# ---------------------------------------------------------------
# STEP 3: Matches - flatten the nested columns
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using select/col/alias from Notebook 01 and dot notation for
# structs from Notebook 02. We ran it and checked the output ourselves.
# ---------------------------------------------------------------
# Fields like competition and home_team are structs (Lecture 07,
# complex types), so we reach inside them with dot notation and
# rename each one with alias() to a simple column name.

matches = matches_raw.select(
    col("match_id"),
    col("match_date"),
    col("competition.competition_name").alias("competition"),
    col("competition.country_name").alias("comp_country"),
    col("season.season_name").alias("season"),
    col("competition_stage.name").alias("stage"),             # e.g. Group Stage, Final
    col("home_team.home_team_name").alias("home_team"),
    col("away_team.away_team_name").alias("away_team"),
    col("home_team.home_team_gender").alias("gender"),        # male / female
    col("home_score"),
    col("away_score"))

# Check the new flat structure and preview 5 rows
matches.printSchema()
matches.show(5, False)

root
 |-- match_id: long (nullable = true)
 |-- match_date: string (nullable = true)
 |-- competition: string (nullable = true)
 |-- comp_country: string (nullable = true)
 |-- season: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- home_score: long (nullable = true)
 |-- away_score: long (nullable = true)

+--------+----------+-----------+------------+---------+--------------+----------------------+--------------+------+----------+----------+
|match_id|match_date|competition|comp_country|season   |stage         |home_team             |away_team     |gender|home_score|away_score|
+--------+----------+-----------+------------+---------+--------------+----------------------+--------------+------+----------+----------+
|3825739 |2016-01-17|La Liga    |Spain       |2015/2016|Regular Season|Real Madrid           |Sporting Gijón|male  |5         |1       

In [6]:
#Step 4: Matches, checking for missing values 

#Every column count the rows where the column is null 
[(c, matches.where(col(c).isNull()).count()) for c in matches.columns]

#The matches table has no missing values in any of its 11 columns -> 
#no rows need to be dropped or filled 

[('match_id', 0),
 ('match_date', 0),
 ('competition', 0),
 ('comp_country', 0),
 ('season', 0),
 ('stage', 0),
 ('home_team', 0),
 ('away_team', 0),
 ('gender', 0),
 ('home_score', 0),
 ('away_score', 0)]

In [8]:
#Step 5: Matches, check for and remove any duplicates 

number_of_matches_before = matches.count()
number_unique = matches.select("match_id").distinct().count()
print("Rows:", number_of_matches_before)
print("Unique matches:", number_unique)
print("Duplicates:", number_of_matches_before - number_unique)

Rows: 3961
Unique matches: 3961
Duplicates: 0


In [10]:
#Step 6: Convert the match data from text to real dates --> right now stored 
# as text; Spark needs it to be converted 

matches = matches.withColumn("match_date", to_date(col("match_date"), "yyyy-MM-dd"))
print("Number of dates that failed to convert:", matches.where(col("match_date").isNull()).count())
#Show the earliest and latest match dates 
matches.selectExpr("min(match_date)", "max(match_date)").show()
#Confirm match_date is not a string 
matches.printSchema()

#All 3,961 match dates converted cleanly to a data type, covering 1958 to 2025. 

Number of dates that failed to convert: 0


+---------------+---------------+
|min(match_date)|max(match_date)|
+---------------+---------------+
|     1958-06-24|     2025-07-27|
+---------------+---------------+

root
 |-- match_id: long (nullable = true)
 |-- match_date: date (nullable = true)
 |-- competition: string (nullable = true)
 |-- comp_country: string (nullable = true)
 |-- season: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- home_score: long (nullable = true)
 |-- away_score: long (nullable = true)



In [17]:
#Step 7: Checking category values 

#Gender should only male or female 
matches.groupBy("gender").count().show()

#Count the matches in each competition, largest to smallest 
competition_counts = matches.groupBy("competition", "gender").count()

#Sorting this from most to fewest matches then showing 
competition_counts = competition_counts.orderBy(col("count").desc()).show()

#Gender values are male and female only. Men's matches outnumber womens 
#roughly 2:1 (2,651 vs. 1,310). We have to compare the per-match rates rather 
#than the totals. Some competitions like La Liga have a lot of matches and 
#while some competitions have few matches so we cannot do a fair comparison 

+------+-----+
|gender|count|
+------+-----+
|  male| 2651|
|female| 1310|
+------+-----+



+--------------------+------+-----+
|         competition|gender|count|
+--------------------+------+-----+
|             La Liga|  male|  867|
|FA Women's Super ...|female|  457|
|             Ligue 1|  male|  435|
|      Premier League|  male|  418|
|             Serie A|  male|  381|
|              Liga F|female|  240|
|                NWSL|female|  173|
|      FIFA World Cup|  male|  147|
|   Frauen Bundesliga|female|  132|
|       Serie A Women|female|  130|
|   Women's World Cup|female|  116|
| Indian Super league|  male|  115|
|           UEFA Euro|  male|  102|
|       1. Bundesliga|  male|   68|
|   UEFA Women's Euro|female|   62|
|African Cup of Na...|  male|   52|
|        Copa America|  male|   32|
|    Champions League|  male|   18|
| Major League Soccer|  male|    6|
|  UEFA Europa League|  male|    3|
+--------------------+------+-----+
only showing top 20 rows



In [18]:
#Step 8 Remove one-off historic matches (before 2000) 
#The dataset includes few famous old matches that cannot truly 
#describe how a player perfomed over a season 

historic_matches = matches.where(col("match_date") < lit("2001-01-01"))
historic_counts = historic_matches.groupBy("competition").count()
historic_counts.show(50, False)

#Count the number of matches before removing 
matches_before_removing = matches.count()

#Keeping only the matches from 2000 onwards 

matches = matches.where(col("match_date") >= lit("2000-01-01"))

#Count after removing 
matches_after_removing = matches.count()
print("Matches before:", matches_before_removing)
print("Matches after:", matches_after_removing)
print("The number of historic matches removed:", matches_before_removing - matches_after_removing)

+---------------------+-----+
|competition          |count|
+---------------------+-----+
|UEFA Europa League   |3    |
|FIFA World Cup       |19   |
|FIFA U20 World Cup   |1    |
|Champions League     |4    |
|North American League|1    |
|Copa del Rey         |3    |
|Liga Profesional     |2    |
|La Liga              |1    |
|Serie A              |1    |
+---------------------+-----+



Matches before: 3961
Matches after: 3926
The number of historic matches removed: 35


In [19]:
# ---------------------------------------------------------------
# STEP 9: Choose which matches to clean events for
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using isin() and & from Notebook 02 and collect() from Notebook 01.
# We ran it and checked the output ourselves.
# ---------------------------------------------------------------
# TEST_MODE (set in the setup cell) keeps only the 2022 men's and
# 2023 women's World Cups, so the event steps run quickly.

if TEST_MODE:
    chosen = matches.where(col("competition").isin(TEST_COMPETITIONS) & col("season").isin(TEST_SEASONS))
else:
    chosen = matches

# Show what was chosen
chosen_counts = chosen.groupBy("competition", "season", "gender").count()
chosen_counts.show(truncate=False)

# Collect the chosen match IDs into a Python list
# (collect() brings the rows back to the driver; this is safe because
# it's just one small column of IDs)
match_id_rows = chosen.select("match_id").collect()
match_ids = [row["match_id"] for row in match_id_rows]
print("Number of matches selected:", len(match_ids))

+-----------------+------+------+-----+
|competition      |season|gender|count|
+-----------------+------+------+-----+
|FIFA World Cup   |2022  |male  |64   |
|Women's World Cup|2023  |female|64   |
+-----------------+------+------+-----+



Number of matches selected: 128


In [20]:
# ---------------------------------------------------------------
# STEP 10: Events - load the event files for the chosen matches
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using the read pattern from Notebook 01, regexp_extract from
# Notebook 02, and cast from Notebook 01. input_file_name() was not
# covered in class (see note below). We ran it and checked the output.
# ---------------------------------------------------------------

# Turn the list of match IDs into file paths, e.g. .../events/3857276.json
paths = [DATA + "events/" + str(i) + ".json" for i in match_ids]

# Load only those files (multiLine because each file is one big JSON array)
events_raw = spark.read.format("json")\
  .option("multiLine", "true")\
  .load(paths)

# The event files don't contain a match_id, but each file is named after
# its match. input_file_name() gives the file each row came from (not in
# the lecture notebooks), and regexp_extract (Notebook 02) pulls out the
# number before ".json". cast("int") (Notebook 01) turns it into a number.
events_raw = events_raw.withColumn(
    "match_id",
    regexp_extract(input_file_name(), r"(\d+)\.json$", 1).cast("int"))

# How many columns the raw events have (lots of nesting)
print("Number of top-level columns:", len(events_raw.columns))

Number of top-level columns: 42


In [21]:
# ---------------------------------------------------------------
# STEP 11: Events - keep only the columns we need; raw data has 
#hundreds of fields once counted of everything nested inside. 
#only pulling out 20 that matter for the project 
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using selectExpr and "AS" renaming from Notebook 01, dot notation for
# structs and [0] for arrays from Notebook 02. We ran it and checked
# the output ourselves.
# ---------------------------------------------------------------
# - Dot notation reaches inside nested columns, e.g. "type.name"
#   (Notebook 02: "complex.Description")
# - [0] and [1] pick items from an array: location is [x, y]
#   (Notebook 02: "array_col[0]")
# - Backticks around `index` and `second` because those words are
#   also SQL keywords (Notebook 01 uses backticks for column names)

events = events_raw.selectExpr(
    # Identifiers and timing
    "id as event_id",
    "match_id",
    "`index` as event_index",
    "period",                                   # 1 = first half, 2 = second half, 5 = shootout
    "minute",
    "`second` as second",

    # What happened and who did it
    "type.name as event_type",                  # Pass, Carry, Shot, Pressure...
    "team.name as team",
    "player.id as player_id",
    "player.name as player",
    "position.name as position",

    # Where it happened (pitch is 120 x 80)
    "location[0] as x",
    "location[1] as y",
    "under_pressure",

    # Passes: where they ended and whether they were completed
    "pass.end_location[0] as pass_end_x",
    "pass.end_location[1] as pass_end_y",
    "pass.outcome.name as pass_outcome",

    # Carries (running with the ball): where they ended
    "carry.end_location[0] as carry_end_x",
    "carry.end_location[1] as carry_end_y",

    # Dribbles and shots
    "dribble.outcome.name as dribble_outcome",
    "shot.statsbomb_xg as shot_xg",             # expected goals: chance quality from 0 to 1
    "shot.outcome.name as shot_outcome")

events.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- event_index: long (nullable = true)
 |-- period: long (nullable = true)
 |-- minute: long (nullable = true)
 |-- second: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player_id: long (nullable = true)
 |-- player: string (nullable = true)
 |-- position: string (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- under_pressure: boolean (nullable = true)
 |-- pass_end_x: double (nullable = true)
 |-- pass_end_y: double (nullable = true)
 |-- pass_outcome: string (nullable = true)
 |-- carry_end_x: double (nullable = true)
 |-- carry_end_y: double (nullable = true)
 |-- dribble_outcome: string (nullable = true)
 |-- shot_xg: double (nullable = true)
 |-- shot_outcome: string (nullable = true)



In [22]:
# ---------------------------------------------------------------
# STEP 12: Events - save a flat parquet copy and reload it
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using write.save and read.load for parquet from the PySpark SQL
# cheat sheet. We ran it and checked the output ourselves.
# ---------------------------------------------------------------
# Without this, every count() below would re-read all the JSON files.
# mode="overwrite" lets us rerun this cell without an "already exists" error.

events.write.save(OUT + "events_flat", format="parquet", mode="overwrite")

# Reload from parquet: fast from here on
events = spark.read.load(OUT + "events_flat")

# Starting point for the event cleaning
events_raw_count = events.count()
print("Number of events loaded:", events_raw_count)

# Preview a few rows (only some columns, so the table fits on screen)
events.select("match_id", "minute", "event_type", "team", "player", "x", "y").show(10, False)

#The 128 test matches contain 460,755 events (about  3,600 per match).

Number of events loaded: 460755


+--------+------+-------------+-------+----------------+----+----+
|match_id|minute|event_type   |team   |player          |x   |y   |
+--------+------+-------------+-------+----------------+----+----+
|3869420 |0     |Starting XI  |Croatia|NULL            |NULL|NULL|
|3869420 |0     |Starting XI  |Brazil |NULL            |NULL|NULL|
|3869420 |0     |Half Start   |Croatia|NULL            |NULL|NULL|
|3869420 |0     |Half Start   |Brazil |NULL            |NULL|NULL|
|3869420 |0     |Pass         |Croatia|Luka Modrić     |60.0|40.0|
|3869420 |0     |Ball Receipt*|Croatia|Marcelo Brozović|44.6|38.4|
|3869420 |0     |Carry        |Croatia|Marcelo Brozović|44.6|38.4|
|3869420 |0     |Pass         |Croatia|Marcelo Brozović|46.8|38.4|
|3869420 |0     |Ball Receipt*|Croatia|Luka Modrić     |63.7|37.4|
|3869420 |0     |Carry        |Croatia|Luka Modrić     |63.7|37.4|
+--------+------+-------------+-------+----------------+----+----+
only showing top 10 rows



In [24]:
#Step 13 Looking at which event types exist and how often they exist, 
#also counting the missing values in each column 

event_type_counts = events.groupBy("event_type").count()
event_type_counts = event_type_counts.orderBy(col("count").desc())
event_type_counts.show()

#Checking missing values 
[(c, events.where(col(c).isNull()).count()) for c in events.columns]

#Passes (128,295) and ball receipts (120,335) and carries (100,536) make up about 
#76% of the 460,755 events, while shots are rare 
#data also contains non-action events like substitutions and injuries, 
#and columns that are empty

+---------------+------+
|     event_type| count|
+---------------+------+
|           Pass|128295|
|  Ball Receipt*|120335|
|          Carry|100536|
|       Pressure| 40997|
|  Ball Recovery| 14286|
|           Duel|  9495|
|      Clearance|  5965|
|          Block|  5419|
|     Miscontrol|  4162|
|        Dribble|  3956|
|    Goal Keeper|  3863|
|   Dispossessed|  3416|
| Foul Committed|  3194|
|           Shot|  3172|
|       Foul Won|  3015|
|   Interception|  2762|
|  Dribbled Past|  2274|
|   Substitution|  1081|
|Injury Stoppage|   840|
|          50/50|   607|
+---------------+------+
only showing top 20 rows



[('event_id', 0),
 ('match_id', 0),
 ('event_index', 0),
 ('period', 0),
 ('minute', 0),
 ('second', 0),
 ('event_type', 0),
 ('team', 0),
 ('player_id', 2253),
 ('player', 2253),
 ('position', 2253),
 ('x', 4262),
 ('y', 4262),
 ('under_pressure', 370009),
 ('pass_end_x', 332460),
 ('pass_end_y', 332460),
 ('pass_outcome', 433063),
 ('carry_end_x', 360219),
 ('carry_end_y', 360219),
 ('dribble_outcome', 456799),
 ('shot_xg', 457583),
 ('shot_outcome', 457583)]

In [26]:
#Step 14 checking for duplicate events 

number_of_events_before = events.count()
number_of_unique_events = events.select("event_id").distinct().count()

print("Rows:", number_of_events_before)
print("Number of unique events:", number_of_unique_events)
print("Number of duplicates:", number_of_events_before - number_of_unique_events)

Rows: 460755
Number of unique events: 460755
Number of duplicates: 0


In [36]:
#Step 15 Removing penalty shootouts: they are not considered normal play 
#they add nothing to shots and xG to the player's numbers; 
#have nothing to do with they played during the game 

#period 1 (first half), period 2 (second half), period (3-4) extra time
#period 5 (shootout)


# Start fresh from the saved flat copy so the counts are accurate (From AI)
events = spark.read.load(OUT + "events_flat")


period_count = events.groupBy("period").count()
period_count = period_count.orderBy("period")
period_count.show()

#Count before removing 
events_before_shootouts = events.count()

#keeping every period expect the shootout

events = events.where(col("period") != 5)

events_after_shootout = events.count()
print("Events before:", events_before_shootouts)
print("Events after:", events_after_shootout)
print("Shootout events removed:", events_before_shootouts - events_after_shootout)

+------+------+
|period| count|
+------+------+
|     1|229086|
|     2|222004|
|     3|  4859|
|     4|  4601|
|     5|   205|
+------+------+



Events before: 460755
Events after: 460550
Shootout events removed: 205


In [37]:
#Step 16 Keeping only player actions
#Removing events with no players and admin events (substitutions, 
#injury stoppages, tactical shifts)

#Utilized AI to simplify typing this chunk of code 
#Events that are not "player ball" actions 
admin_types = ["Starting XI", "Half Start", "Half End", "Tactical Shift", "Substitution",
               "Player On", "Player Off", "Injury Stoppage", "Referee Ball-Drop",
               "Camera On", "Camera off", "Own Goal For", "Bad Behaviour"]
events_before_actions = events.count()

#keep rows that have a player and drop the admin events 

events = events.where(~col("player_id").isNull())
events = events.where(~col("event_type").isin(admin_types))

events_after_actions = events.count()
print("Events before:", events_before_actions)
print("Events after:", events_after_actions)
print("Non-player and admin events removed:",events_before_actions - events_after_actions)

Events before: 460550
Events after: 455937
Non-player and admin events removed: 4613


In [38]:
# ---------------------------------------------------------------
# STEP 17: Events - fill nulls that actually mean something
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using na.fill with a dictionary from Notebook 02 and when/otherwise
# We ran it and checked the output.
# ---------------------------------------------------------------
# WHY THIS STEP EXISTS
# StatsBomb only writes some fields down when something "happens."
# If nothing happened, it leaves the field empty (null). So these
# nulls are not missing data; they carry meaning:
#   - under_pressure: StatsBomb writes True when a defender was
#     pressuring the player. If there was no pressure, it writes
#     nothing. So null really means False.
#   - pass_outcome: StatsBomb only writes an outcome when a pass
#     FAILED (e.g. "Incomplete", "Out"). If the pass reached a
#     teammate, it writes nothing. So for a pass, null means "Complete".
# If we left these as null, later counts would be wrong. For example,
# "completed passes" would count 0, because no row says "Complete".

# ---------------------------------------------------------------
# BEFORE: count the nulls so we can prove the fix worked
# ---------------------------------------------------------------
# where(...) keeps only rows matching the condition, count() counts them.
# isNull() is True when the cell is empty.
print("under_pressure nulls before:", events.where(col("under_pressure").isNull()).count())

# Two conditions joined with & ("and"), each wrapped in its own ( ):
#   (col("event_type") == "Pass")   -> the row is a pass
#   col("pass_outcome").isNull()    -> and its outcome is empty
# So this counts completed passes that currently have no label.
print("Passes with no outcome before:",
      events.where((col("event_type") == "Pass") & col("pass_outcome").isNull()).count())

# ---------------------------------------------------------------
# FIX 1: under_pressure -> fill empty cells with False
# ---------------------------------------------------------------
# na.fill({...}) fills null cells. The dictionary says WHICH column
# gets WHICH value: {"column name": value to put in empty cells}.
# Only the under_pressure column is touched; all other columns stay
# exactly as they were. Rows that already say True are not changed.
# Same pattern as Notebook 02:
#   df.na.fill({"CustomerID": 0.0, "Description": "No Value"})
events = events.na.fill({"under_pressure": False})

# ---------------------------------------------------------------
# FIX 2: pass_outcome -> fill with "Complete", but ONLY for passes
# ---------------------------------------------------------------
# Why not just na.fill({"pass_outcome": "Complete"})? Because that would
# fill EVERY empty pass_outcome, including on shots, carries, tackles...
# Those aren't passes, so labelling them "Complete" would be wrong.
#
# Instead we use when/otherwise, which works like an IF / ELSE for
# each row (cheat sheet, "Queries -> When"):
#
#   withColumn("pass_outcome", ...)
#       rebuild the pass_outcome column, row by row, using the rule below
#
#   F.when( CONDITION , "Complete" )
#       IF the row is a pass AND its outcome is empty
#       THEN put "Complete"
#
#   .otherwise( col("pass_outcome") )
#       ELSE keep whatever was already there:
#         - failed passes keep "Incomplete", "Out", etc.
#         - non-pass events keep their null (they have no pass outcome)
events = events.withColumn(
    "pass_outcome",
    F.when((col("event_type") == "Pass") & col("pass_outcome").isNull(), "Complete")
     .otherwise(col("pass_outcome")))

# ---------------------------------------------------------------
# AFTER: check that both fixes worked
# ---------------------------------------------------------------
# Should print 0: every empty under_pressure is now False
print("under_pressure nulls after:", events.where(col("under_pressure").isNull()).count())

# Look only at passes, count how many have each outcome, biggest first.
# Expect "Complete" at the top (most passes succeed), then the failure
# types: Incomplete, Out, Pass Offside, etc. There should be NO null
# row any more, because every pass now has an outcome.
pass_outcomes = events.where(col("event_type") == "Pass").groupBy("pass_outcome").count()
pass_outcomes = pass_outcomes.orderBy(col("count").desc())
pass_outcomes.show(truncate=False)

under_pressure nulls before: 365201


Passes with no outcome before: 100603
under_pressure nulls after: 0


+----------------+------+
|pass_outcome    |count |
+----------------+------+
|Complete        |100603|
|Incomplete      |23838 |
|Out             |2688  |
|Unknown         |705   |
|Pass Offside    |389   |
|Injury Clearance|72    |
+----------------+------+



In [39]:
# ---------------------------------------------------------------
# STEP 18: Events - check that all coordinates are on the pitch
# ---------------------------------------------------------------
# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using | ("or") and ~ ("not") from Notebook 02 / Lecture 07 and
# describe() from Notebook 02. We ran it and checked the output.
# ---------------------------------------------------------------
# HOW THE PITCH WORKS
# StatsBomb draws every pitch the same way, attacking left to right:
#   x goes from 0 (own goal line) to 120 (opponent's goal line)
#   y goes from 0 to 80 (one touchline to the other)
# So a valid location always has 0 <= x <= 120 and 0 <= y <= 80.

# ---------------------------------------------------------------
# Build the "off the pitch" condition
# ---------------------------------------------------------------
# | means "or" (Notebook 02: priceFilter | descripFilter).
# A row is off the pitch if ANY of these four things is true.
# Each comparison sits in its own ( ) so Python reads them correctly.
off_pitch = (col("x") < 0) | (col("x") > 120) | (col("y") < 0) | (col("y") > 80)

# How many events have a location at all (a few types, e.g. some
# Pressure or Ball Receipt events, can have no location)
print("Events with a location:", events.where(~col("x").isNull()).count())

# How many locations are off the pitch (we expect 0)
print("Events off the pitch:", events.where(off_pitch).count())

# ---------------------------------------------------------------
# Remove off-pitch events (if any)
# ---------------------------------------------------------------
# Keep a row if it is NOT off the pitch  (~off_pitch)
# OR if it has no location at all       (col("x").isNull())
# Why the second part? When x is empty, Spark can't decide whether
# it's on or off the pitch, so ~off_pitch comes out as "unknown" and
# where() would quietly DROP the row. Adding "or x is empty" keeps
# those rows, since having no location is not an error.
events_before_pitch = events.count()
events = events.where(~off_pitch | col("x").isNull())
events_after_pitch = events.count()

print("Events before:", events_before_pitch)
print("Events after:", events_after_pitch)
print("Off-pitch events removed:", events_before_pitch - events_after_pitch)

# ---------------------------------------------------------------
# Summary statistics as a final check (Notebook 02: describe)
# ---------------------------------------------------------------
# For each column: count, mean, stddev, min, max.
# Check that every min is >= 0 and every max is <= 120 (x) / 80 (y).
# shot_xg should be between 0 and 1, since it's a probability.
events.describe(["x", "y", "pass_end_x", "carry_end_x", "shot_xg"]).show()

#Every event has a valid location on the 120 x 80 pitch except one recording 
#error, which were removed 

Events with a location: 455937


Events off the pitch: 1


Events before: 455937
Events after: 455936
Off-pitch events removed: 1


26/11/02 04:57:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-----------------+------------------+-------------------+
|summary|                 x|                 y|       pass_end_x|       carry_end_x|            shot_xg|
+-------+------------------+------------------+-----------------+------------------+-------------------+
|  count|            455936|            455936|           128295|            100536|               3087|
|   mean| 58.00938903705629|39.895081546529916|63.60426984683731|60.454314872284854|0.10430254774522182|
| stddev|28.865389734549307|   23.609655944036| 28.0987082864823|27.292429625403475|0.14582961096353755|
|    min|               0.1|               0.1|              0.1|               0.1|             1.8E-4|
|    max|             120.0|              80.0|            120.0|             120.0|          0.9280499|
+-------+------------------+------------------+-----------------+------------------+-------------------+



In [42]:
#Step 19 adding competitino, season, gender, and date to each event 
#through matching on match_id 

events.createOrReplaceTempView("events")
matches.createOrReplaceTempView("matches")

events_clean =spark.sql("""SELECT events.*, matches.competition, matches.season,
matches.gender, matches.match_date FROM events JOIN matches ON events.match_id = matches.match_id""")

print("Events before:", events.count())
print("Events after:", events_clean.count())

Events before: 455936


Events after: 455936


In [44]:
#Step 20 Summarizing the clean events 

print("Clean events:", events_clean.count())
print("Columns:", len(events_clean.columns))

events_clean.printSchema()

summary = events_clean.groupBy("competition", "gender").agg(expr("count(distinct match_id) as matches"), expr("count(distinct player_id) as players"), expr("count(event_id) as events"))

summary.show(truncate=False)

Clean events: 455936
Columns: 26
root
 |-- event_id: string (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- event_index: long (nullable = true)
 |-- period: long (nullable = true)
 |-- minute: long (nullable = true)
 |-- second: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player_id: long (nullable = true)
 |-- player: string (nullable = true)
 |-- position: string (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- under_pressure: boolean (nullable = false)
 |-- pass_end_x: double (nullable = true)
 |-- pass_end_y: double (nullable = true)
 |-- pass_outcome: string (nullable = true)
 |-- carry_end_x: double (nullable = true)
 |-- carry_end_y: double (nullable = true)
 |-- dribble_outcome: string (nullable = true)
 |-- shot_xg: double (nullable = true)
 |-- shot_outcome: string (nullable = true)
 |-- competition: string (nullable = true)
 |-- season: string (nullable = true

+-----------------+------+-------+-------+------+
|competition      |gender|matches|players|events|
+-----------------+------+-------+-------+------+
|FIFA World Cup   |male  |64     |679    |232265|
|Women's World Cup|female|64     |616    |223671|
+-----------------+------+-------+-------+------+



In [45]:
#Step 21 Saving the clean tables and allowing teammates to not redo it 

# AI disclosure: this cell was drafted with help from Claude (Anthropic),
# using write.save for parquet from the PySpark SQL cheat sheet
# (same as Step 12). We ran it and checked the output ourselves.
# ---------------------------------------------------------------
# Saved to the bucket, so the files stay even after the cluster is deleted.
# mode="overwrite" lets us rerun this cell without an "already exists" error.

matches.write.save(OUT + "matches_clean", format="parquet", mode="overwrite")
events_clean.write.save(OUT + "events_clean", format="parquet", mode="overwrite")

print("Saved to:", OUT)


Saved to: gs://ba843_michaels_part/statsbomb/clean/test/
